# LangChain Agent Benchmark 02: RAG, Embeddings, Chunking, Vector Stores, and Retrieval

This notebook compares retrieval settings using the text from `docs/paper.pdf`. It uses LangChain's document loader, text splitter, embedding, vector store, and retrieval interfaces directly.
Careful: RAG can only be used on unstructured text, ie. papers, documentation, etc., not tables or databases. RAG adaptations to tabular data exist but are not state of the art RAG. Reliable methods to query structured text follow in notebook 03.


In [1]:
import os
import time
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown

from langchain.chat_models import init_chat_model
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

if not os.getenv("GOOGLE_API_KEY"):
    print("Set GOOGLE_API_KEY before running the examples.")

PDF_PATH = Path("../docs/paper.pdf")
study_pdf = PyPDFLoader(str(PDF_PATH)).load()

for doc in study_pdf:
    doc.metadata["source"] = str(PDF_PATH)
    doc.metadata["doc_type"] = "paper_pdf"
    doc.metadata["paper"] = PDF_PATH.name


def format_sources(docs):
    sources = sorted({doc.metadata.get("source", "unknown") for doc in docs})
    return ", ".join(sources) if sources else "None"


def format_contexts(docs):
    return "\n\n".join(
        f"{rank}. {' '.join(doc.page_content.split())}"
        for rank, doc in enumerate(docs, start=1)
    )


/var/folders/ch/9sqn9qpd5cbfpy3vzf372t9c0000gp/T/ipykernel_59740/1189494441.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Set GOOGLE_API_KEY before running the examples.


## 1. Embedding models

Embedding model controls how documents and queries are represented for semantic search, which affects retrieval precision and relevance. The models below differ by size, training objective, language coverage, and domain specialization.

These Hugging Face embedding models do not use streaming or usage reporting because they are not chat/completion models and they are not making a paid API call. They just turn text into vectors.

The embedding models we will test are:
- SentenceTransformers MiniLM general small: Small, fast, general-purpose sentence encoder.
- SentenceTransformers Multi-QA retrieval tuned: Trained on question-answer pairs for semantic search.
- PubMedBERT biomedical: Biomedical literature model tuned on PubMed title-abstract pairs.

**Reflection Prompts**
- Compare the top hits from the different embeddings: do they retrieve the same passages or different evidence?
- Evaluate the trade-off between a larger model, execution time, and result relevance.
- Ask whether a general-purpose embedding is enough for a specialized biology paper.


In [17]:
query = "CRISPLD2 glucocorticoid dexamethasone airway smooth muscle"
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
splits = text_splitter.split_documents(study_pdf)

embedding_models = [
    {
        "label": "MiniLM general small",
        "model_name": "sentence-transformers/all-MiniLM-L6-v2",
        "description": "Small, fast, general-purpose sentence encoder.",
    },
    {
        "label": "Multi-QA retrieval tuned",
        "model_name": "sentence-transformers/multi-qa-mpnet-base-dot-v1",
        "description": "Trained on question-answer pairs for semantic search.",
    },
    {
        "label": "PubMedBERT biomedical",
        "model_name": "NeuML/pubmedbert-base-embeddings",
        "description": "Biomedical literature model tuned on PubMed title-abstract pairs.",
    },
]

rows = []
for spec in embedding_models:
    row = {"embedding_model": spec["model_name"]}

    try:
        start = time.perf_counter()
        embedding_model = HuggingFaceEmbeddings(
            model_name=spec["model_name"],
            model_kwargs={"device": "cpu"},
            encode_kwargs={"normalize_embeddings": True},
        )
        dimension = len(embedding_model.embed_query("dimension check"))
        vector_store = InMemoryVectorStore.from_documents(documents=splits, embedding=embedding_model)
        hits_with_scores = vector_store.similarity_search_with_score(query, k=3)

        row.update({
            "dimension": dimension,
            "seconds": round(time.perf_counter() - start, 2),
            "top_hits": "\n\n".join(
                f"{rank}. score={score:.3f}\n{doc.page_content[:140]}"
                for rank, (doc, score) in enumerate(hits_with_scores, start=1)
            ),
            "source": format_sources([doc for doc, score in hits_with_scores]),
        })
    except Exception as exc:
        row.update({
            "dimension": None,
            "seconds": None,
            "top_hits": "",
            "source": "",
            "error": repr(exc),
        })

    rows.append(row)

df = pd.DataFrame(rows)
df.style.set_properties(
    subset=["top_hits", "source"],
    **{
        "white-space": "pre-wrap",
        "text-align": "left",
        "min-width": "350px",
        "vertical-align": "top",
    },
)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

,embedding_model,dimension,seconds,top_hits,source
0,sentence-transformers/all-MiniLM-L6-v2,384,12.700000,1. score=0.745 RNA-Seq Transcriptome Profiling Identifies CRISPLD2 as a Glucocorticoid Responsive Gene that Modulates Cytokine Function in Airway Smooth Mu 2. score=0.721 offer a comprehensive view of the effect of a glucocorticoid on the ASM transcriptome and identify CRISPLD2 as an asthma pharmacogenetics ca 3. score=0.696 Glucocorticoid-induced Changes in Gene Expression of Airway Smooth Muscle in Patients with Asthma. Am J Respir Crit Care Med 187: 1076–1084.,../docs/paper.pdf
1,sentence-transformers/multi-qa-mpnet-base-dot-v1,768,51.310000,"1. score=0.698 RNA-Seq Transcriptome Profiling Identifies CRISPLD2 as a Glucocorticoid Responsive Gene that Modulates Cytokine Function in Airway Smooth Mu 2. score=0.660 mainstay therapy for asthma because they exert anti-inflammatory effects in multiple lung tissues, including the airway smooth muscle (ASM). 3. score=0.625 doi:10.1371/journal.pone.0099625.g002 CRISPLD2 Is a Glucocorticoid Responsive Gene in ASM PLOS ONE | www.plosone.org 4 June 2014 | Volume 9",../docs/paper.pdf
2,NeuML/pubmedbert-base-embeddings,768,37.220000,"1. score=0.631 mainstay therapy for asthma because they exert anti-inflammatory effects in multiple lung tissues, including the airway smooth muscle (ASM). 2. score=0.622 DEX, and the GSE13168 study found that the differential CRISPLD2 expression was strongest when ASM cells were treated with a GC (i.e. flutic 3. score=0.613 Glucocorticoid-induced Changes in Gene Expression of Airway Smooth Muscle in Patients with Asthma. Am J Respir Crit Care Med 187: 1076–1084.",../docs/paper.pdf


Notice that the 500 characters of each retrieved chunks are truncated for presentation.

## 2. Retrieval (RAG)

Retrieval controls whether the answer is generated from pretrained knowledge only, or from chunks of retrieved context provided.
The parameter k controls the top k number of similar chunks retrieved. Less chunks give the mdoel less context so answers are focused and cheaper, more chunks have a better chance of finding the answer but may be dispersive, longer, and more expensive.

**Reflection Prompts**
- Compare the answer without RAG to the context-based answers: where does it improve, and where does it get worse?
- Observe when the model says it does not know and discuss whether this is desirable behavior.
- Verify whether the retrieved sources really contain the evidence needed for the answer.


In [18]:
question = "What did dexamethasone treatment do to CRISPLD2 expression in airway smooth muscle cells?"
model = init_chat_model("gemini-3.1-flash-lite", model_provider="google_genai", temperature=0)

rag_embedding_models = [
    {
        "label": "MiniLM general small",
        "model_name": "sentence-transformers/all-MiniLM-L6-v2",
    },
    {
        "label": "Multi-QA retrieval tuned",
        "model_name": "sentence-transformers/multi-qa-mpnet-base-dot-v1",
    },
]

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
splits = text_splitter.split_documents(study_pdf)

for embedding_spec in rag_embedding_models:
    embedding_model = HuggingFaceEmbeddings(
        model_name=embedding_spec["model_name"],
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": True},
    )
    vector_store = InMemoryVectorStore.from_documents(documents=splits, embedding=embedding_model)

    prompt_specs = [
        {"mode": "no_rag", "prompt": question, "sources": format_sources([])},
    ]

    for k in [2, 6]:
        retrieved_docs = vector_store.similarity_search(question, k=k)
        context = chr(10).join([doc.page_content for doc in retrieved_docs])
        prompt_specs.append({
            "mode": f"rag_k_{k}",
            "prompt": f"Answer using only this context. If the answer is absent, say you do not know.\n\n{context}\n\nQuestion: {question}",
            "sources": format_sources(retrieved_docs),
        })

    for spec in prompt_specs:
        start = time.perf_counter()
        response = model.invoke(spec["prompt"])
        answer = response.content if isinstance(response.content, str) else str(response.content)
        markdown_text = chr(10).join([
            f"### {embedding_spec['label']} | {spec['mode']}",
            f"`{embedding_spec['model_name']}`",
            "",
            answer,
            "",
            "**Sources**",
            spec["sources"],
        ])
        display(Markdown(markdown_text))
        print("seconds:", round(time.perf_counter() - start, 3))
        time.sleep(3)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

### MiniLM general small | no_rag
`sentence-transformers/all-MiniLM-L6-v2`

[{'type': 'text', 'text': "Research into the effects of corticosteroids on airway smooth muscle (ASM) cells has shown that **dexamethasone significantly increases (upregulates) the expression of *CRISPLD2*** (Cysteine-Rich Secretory Protein LCCL Domain Containing 2).\n\nHere are the key findings regarding this interaction:\n\n*   **Upregulation:** Studies, most notably those investigating the genetic susceptibility to asthma and the response to corticosteroids, have identified *CRISPLD2* as a highly responsive gene to glucocorticoids. Treatment with dexamethasone leads to a robust induction of *CRISPLD2* mRNA and protein levels in ASM cells.\n*   **Anti-inflammatory Role:** The upregulation of *CRISPLD2* by dexamethasone is considered a protective mechanism. *CRISPLD2* has been shown to inhibit the production of pro-inflammatory cytokines (such as IL-6 and CXCL8) in ASM cells. Therefore, the induction of this gene is one of the pathways through which corticosteroids exert their anti-inflammatory effects in the airway.\n*   **Genetic Variation:** Research has highlighted that single nucleotide polymorphisms (SNPs) in the *CRISPLD2* gene can influence the magnitude of this upregulation. Individuals with certain genetic variants may show a diminished *CRISPLD2* response to dexamethasone, which has been linked to poorer asthma control and reduced responsiveness to inhaled corticosteroid therapy.\n*   **Context in Asthma:** Because *CRISPLD2* acts as a negative regulator of inflammation, its induction by dexamethasone helps stabilize the airway environment. When this induction is impaired (due to genetic factors or other signaling interference), the ASM cells may remain in a more pro-inflammatory state, contributing to airway hyperresponsiveness.\n\nIn summary, dexamethasone acts as a potent inducer of *CRISPLD2* in airway smooth muscle cells, and this upregulation is a critical component of the drug's ability to suppress airway inflammation.", 'extras': {'signature': 'EjQKMgERTTIPo6Qt/VpVKcI4jvX8ZJBy6VecqWcYlU6QcQ9pdeOOtzOxGI3WBTKWrbZ733/J'}}]

**Sources**
None

seconds: 2.081


### MiniLM general small | rag_k_2
`sentence-transformers/all-MiniLM-L6-v2`

[{'type': 'text', 'text': 'I do not know.', 'extras': {'signature': 'EjQKMgERTTIPTW2QHyeaywFwirNrltV+/ScQlTE1L2+gGT4bWYQ3lI2W0QUff4I+vqJj0BEW'}}]

**Sources**
../docs/paper.pdf

seconds: 0.463


...

seconds: 2.039


### Multi-QA retrieval tuned | rag_k_2
`sentence-transformers/multi-qa-mpnet-base-dot-v1`

[{'type': 'text', 'text': 'Dexamethasone treatment significantly increased CRISPLD2 mRNA and protein expression in airway smooth muscle cells.', 'extras': {'signature': 'EjQKMgERTTIPyMzgRC3/w/97NbJ2kEap6p6v63gxFPFS/ONhGeBOqlKKPfHlKwtsNE7jHq0t'}}]

**Sources**
../docs/paper.pdf

seconds: 0.544


### Multi-QA retrieval tuned | rag_k_6
`sentence-transformers/multi-qa-mpnet-base-dot-v1`

[{'type': 'text', 'text': 'Dexamethasone treatment significantly increased CRISPLD2 mRNA and protein expression in airway smooth muscle cells.', 'extras': {'signature': 'EjQKMgERTTIPlVNffzP8MiXPTaPB3AjGGmDby4k1V/U6I9/yNywCnbg+fGWTm4xxcMLdw+5Y'}}]

**Sources**
../docs/paper.pdf

seconds: 0.427


Notice that when using a small embedding model, RAG performs worse than no RAG in the sense that it "does not know" (because it is unable to retrieve a relevant chunk), however it still puts guards into place with respect to no RAG, which still answers from its pretrained knowledge. In what situation would this guardrail behaviour be useful?

## 3. Chunk size

Chunk size controls the amount of text in each indexed unit, balancing complete context against retrieval specificity.

In notebook 01 section 5, we experimented with context, meaning understanding how much context the LLM could actually see in the prompt to base its answer. That differed from RAG as no documents are searched or retrieved, RAG's role is to automates the context-selection step. 

**Reflection Prompts**
- Observe how the retrieved context changes when chunks are small or large.
- Look for examples where a small chunk loses information needed to answer the question.
- Evaluate whether large chunks introduce noise that can confuse generation.


In [19]:
query = "What RNA-Seq quality control or alignment metrics are reported?"
rows = []
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={"normalize_embeddings": True},
)

for size in [200, 500, 1000, 2000]:
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=100)
    splits = text_splitter.split_documents(study_pdf)
    vector_store = InMemoryVectorStore.from_documents(documents=splits, embedding=embedding_model)
    hits = vector_store.similarity_search(query, k=3)
    rows.append({"chunk_size": size, "n_chunks": len(splits), "top_context": format_contexts(hits)})

df = pd.DataFrame(rows)
df.style.set_properties(
    subset=["top_context"],
    **{"white-space": "pre-wrap", "text-align": "left", "min-width": "650px"},
)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## 4. Chunk overlap

Chunk overlap controls how much text is repeated across adjacent chunks. The overlap helps to avoid losing meaning when an important sentence or concept sits right on the boundary between two chunks. The potential downsides are increasing duplicate retrieval.

```text
Original document

────────────────────────────────────────────────────────────────────────────▶

Split into overlapping chunks

Chunk 1 (500 characters)
[====================++++++++++]
                    100-char overlap

Chunk 2 (500 characters)
          [++++++++++====================]
                              100-char overlap

Chunk 3 (500 characters)
                    [++++++++++====================]

Legend
====================  New text (400 characters)
++++++++++            Overlap from previous chunk (100 characters)
```

**Reflection Prompts**
- Compare whether overlap helps avoid splitting important information across two chunks.
- Observe the cost in terms of number of chunks and possible duplication in the results.
- Decide which overlap you would choose for scientific texts with dense methods and results.


In [21]:
query = "What treatment protocol was used for dexamethasone in airway smooth muscle cells?"
rows = []

for overlap in [0, 100, 300]:
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=overlap)
    splits = text_splitter.split_documents(study_pdf)
    vector_store = InMemoryVectorStore.from_documents(documents=splits, embedding=embedding_model)
    hits = vector_store.similarity_search(query, k=4)
    rows.append({"chunk_overlap": overlap, "n_chunks": len(splits), "top_context": format_contexts(hits)})

df = pd.DataFrame(rows)
df.style.set_properties(
    subset=["top_context"],
    **{"white-space": "pre-wrap", "text-align": "left", "min-width": "650px"},
)


,chunk_overlap,n_chunks,top_context
0,0,158,"1. augment relengthening of contracted airway smooth muscle: potential additional mechanism of benefit in asthma. Eur Respir J 32: 1224–1230. 10. Lamas AM, Leon OG, Schleimer RP (1991) Glucocorticoids inhibit eosinophil responses to granulocyte-macrophage colony-stimulating factor. J Immunol 147: 254–259. 11. Wallen N, Kita H, Weiler D, Gleich GJ (1991) Glucocorticoids inhibit cytokine- mediated eosinophil survival. J Immunol 147: 3490–3495. 2. levels increased in response to DEX, levels decreased in A549 cells according to our experiments [Figure S5B] and were not significantly changed after 100 nM DEX treatment for 1 hour in the Reddy et al [12] study. In a study using BEAS-2B cells, CRISPLD2 expression was induced by both a glucocorticoid (fluticasone) and a long-acting beta-agonist (formoterol) [38]. Further studies are required to understand the cell-specific expression of CRISPLD2 and its interactions with other asthma 3. pharmacogenetics candidate gene that regulates anti-inflammatory effects of glucocorticoids in the ASM. Citation: Himes BE, Jiang X, Wagner P, Hu R, Wang Q, et al. (2014) RNA-Seq Transcriptome Profiling Identifies CRISPLD2 as a Glucocorticoid Responsive Gene that Modulates Cytokine Function in Airway Smooth Muscle Cells. PLoS ONE 9(6): e99625. doi:10.1371/journal.pone.0099625 Editor: Jan Peter Tuckermann, University of Ulm, Germany 4. studies have measured the effect of GCs on ASM cells using in vitro models where human ASM cells were stimulated with dexameth- asone or fluticasone [17,18]. Although both were limited by the inherent biases of microarrays, these studies identified some genes involved in the ASM GC response, with one focusing on validating the function of the KLF15 gene in airway hyperresponsiveness [17] and the other on the overlap between GC and beta-agonist response of the ASM [18]."
1,100,179,"1. PG, et al. (2008) Airway smooth muscle in bronchial tone, inflammation, and remodeling: basic knowledge to clinical relevance. Am J Respir Crit Care Med 177: 248–252. 15. Nelson HS (1995) Beta-adrenergic bronchodilators. N Engl J Med 333: 499– 506. 16. Hargreave FE, Ryan G, Thomson NC, O’Byrne PM, Latimer K, et al. (1981) Bronchial responsiveness to histamine or methacholine in asthma: measurement and clinical significance. J Allergy Clin Immunol 68: 347–355. 2. mainstay therapy for asthma because they exert anti-inflammatory effects in multiple lung tissues, including the airway smooth muscle (ASM). However, the mechanism by which glucocorticoids suppress inflammation in ASM remains poorly understood. Using RNA-Seq, a high-throughput sequencing method, we characterized transcriptomic changes in four primary human ASM cell lines that were treated with dexamethasone—a potent synthetic glucocorticoid (1 mM for 3. Glucocorticoid-induced Changes in Gene Expression of Airway Smooth Muscle in Patients with Asthma. Am J Respir Crit Care Med 187: 1076–1084. 52. Panettieri RA, Murray RK, DePalo LR, Yadvish PA, Kotlikoff MI (1989) A human airway smooth muscle cell line that retains physiological responsiveness. Am J Physiol 256: C329–335. 53. Cooper PR, Mesaros AC, Zhang J, Christmas P, Stark CM, et al. (2010) 20- HETE mediates ozone-induced, neutrophil-independent airway hyper-respon- 4. 8. Trifilieff A, El-Hashim A, Bertrand C (2000) Time course of inflammatory and remodeling events in a murine model of asthma: effect of steroid treatment. Am J Physiol Lung Cell Mol Physiol 279: L1120–1128. 9. Lakser OJ, Dowell ML, Hoyte FL, Chen B, Lavoie TL, et al. (2008) Steroids augment relengthening of contracted airway smooth muscle: potential additional mechanism of benefit in asthma. Eur Respir J 32: 1224–1230."
2,300,341,"1. tions 423: 134–139. 51. Yick CY, Zwinderman AH, Kunst PW, Grunberg K, Mauad T, et al. (2013) Glucocorticoid-induced Changes in Gene Expression of Airway Smooth Muscle in Patients with Asthma. Am J Respir Crit Care Med 187: 1076–1084. 52. Panett

As chunk overlap increases from 0 to 300, the number of chunks rises, improving the chance that boundary-spanning context is preserved. 
The risks of increasing chunk overlap too much are: eventually reducing the number of chunks (not happening here), increasing duplicate retrieval, making embeddings difficult to compute.

## 5. Vector store

Vector store controls where embeddings are stored and queried, affecting speed, usability, persistence, and scalability.

Here is a flowchart of the information flow from the raw PDF to the vector store. 

```text
               PDF
                │
                ▼
        Split into chunks
                │
                ▼
          Embeddings
                │
                ▼
          Vector Store
     ┌────────┼──────────┐
     │        │          │
 InMemory   FAISS     Chroma
   Fast      Fast       Fast
  search    search     search
 (RAM)                + metadata
                      + persistence
                      + filtering
                      + collections
```

**Vector stores**:
- InMemoryVectorStore (LangChain's in-memory vector database): stores vectors only in RAM
- FAISS (Meta): only stores vectors, fast
- Chroma: stores vectors and also metadata, so also does database search on document information

**Reflection Prompts**
- Compare the speed, results, and practicality of the available vector stores.
- Distinguish between an in-memory teaching example and a persistent solution for a real project.
- Think about which metadata you would want to preserve for auditability, reproducibility, and citations.


In [4]:
query = "CRISPLD2 dexamethasone cytokine IL6 IL8"
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
splits = text_splitter.split_documents(study_pdf)
rows = []
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={"normalize_embeddings": True},
)

# 1. InMemoryVectorStore
start = time.perf_counter()
vector_store = InMemoryVectorStore.from_documents(documents=splits, embedding=embedding_model)
hits = vector_store.similarity_search(query, k=3)
rows.append({"store": "InMemoryVectorStore", "seconds": round(time.perf_counter() - start, 4), "source": format_sources(hits)})

# 2. Chroma
try:
    from langchain_chroma import Chroma

    start = time.perf_counter()
    vector_store = Chroma.from_documents(documents=splits, embedding=embedding_model, collection_name="bio_benchmark")
    hits = vector_store.similarity_search(query, k=3)
    rows.append({"store": "Chroma", "seconds": round(time.perf_counter() - start, 4), "source": format_sources(hits)})
except Exception as exc:
    print("Chroma skipped:", exc)

# 3. FAISS
try:
    from langchain_community.vectorstores import FAISS

    start = time.perf_counter()
    vector_store = FAISS.from_documents(documents=splits, embedding=embedding_model)
    hits = vector_store.similarity_search(query, k=3)
    rows.append({"store": "FAISS", "seconds": round(time.perf_counter() - start, 4), "source": format_sources(hits)})
except Exception as exc:
    print("FAISS skipped:", exc)

df = pd.DataFrame(rows)
df.style.set_properties(
    subset=["source"],
    **{"white-space": "pre-wrap", "text-align": "left", "min-width": "650px"},
)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

,store,seconds,source
0,InMemoryVectorStore,3.507600,../docs/paper.pdf
1,Chroma,2.455000,../docs/paper.pdf
2,FAISS,2.840300,../docs/paper.pdf


## 6. Retrieval strategies

Strategies for selecting which documents (or chunks) to return from a vector database after embedding a query. Choose based on whether you want to optimize only for relevance or also for diversity, and include keywords.

In [2]:
QUERY = "How does CRISPLD2 knockdown affect IL1 beta-induced IL6 and IL8 cytokine responses?"
KEYWORD_TERMS = ["CRISPLD2", "IL6", "IL8"]


### i. Similarity search

Similarity search retrieves chunks closest to the query embedding, usually favoring relevance over diversity.


**Reflection Prompts**
- Observe whether the most similar chunks truly answer the question or only share keywords.
- Compare the local relevance of a single chunk with the overall coverage of the evidence.
- Note any redundant passages that limit the variety of the retrieved context.


In [5]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
splits = text_splitter.split_documents(study_pdf)
vector_store = InMemoryVectorStore.from_documents(documents=splits, embedding=embedding_model)

vector_store.similarity_search(QUERY, k=3)


[Document(id='6a1d1484-2697-4383-8218-5ae1f0abdfe1', metadata={'producer': 'Acrobat Distiller 9.0.0 (Windows); modified using iText 5.0.3 (c) 1T3XT BVBA', 'creator': '3B2 Total Publishing System 7.51n/W', 'creationdate': '2014-06-03T10:42:37+08:00', 'title': 'pone.0099625 1..13', 'moddate': '2014-06-04T01:26:28-07:00', 'source': '../docs/paper.pdf', 'total_pages': 13, 'page': 4, 'page_label': '5', 'doc_type': 'paper_pdf', 'paper': 'paper.pdf'}, page_content='IL1b responsiveness of two inflammatory genes (i.e. IL6 and IL8),\nsuggesting that CRISPLD2 may regulate immune response.\nSpecifically, CRISPLD2 may interfere with IL1 b-induced cytokine\nproduction and act to reduce immune response via a negative\nfeedback loop that can be activated by IL1 b. This negative\nfeedback loop may also play a role in cytokine level modulation in\nresponse to DEX treatment, as evidenced by the increased levels of\nIL6 observed when both IL1 b and DEX were administered to'),
 Document(id='020c53c8-01c9-4

### ii. MMR retrieval

MMR (Maximal Marginal Relevance) returns both relevant and diverse chunks


**Reflection Prompts**
- Compare MMR results with similarity search: does useful diversity increase?
- Evaluate whether any less similar chunk adds context that is essential for the answer.
- Discuss when you would prefer diversity over maximum similarity.


In [6]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
splits = text_splitter.split_documents(study_pdf)
vector_store = InMemoryVectorStore.from_documents(documents=splits, embedding=embedding_model)

vector_store.max_marginal_relevance_search(QUERY, k=3)


[Document(id='cb30d102-b764-420e-927a-dd86f7bad3a5', metadata={'producer': 'Acrobat Distiller 9.0.0 (Windows); modified using iText 5.0.3 (c) 1T3XT BVBA', 'creator': '3B2 Total Publishing System 7.51n/W', 'creationdate': '2014-06-03T10:42:37+08:00', 'title': 'pone.0099625 1..13', 'moddate': '2014-06-04T01:26:28-07:00', 'source': '../docs/paper.pdf', 'total_pages': 13, 'page': 4, 'page_label': '5', 'doc_type': 'paper_pdf', 'paper': 'paper.pdf'}, page_content='IL1b responsiveness of two inflammatory genes (i.e. IL6 and IL8),\nsuggesting that CRISPLD2 may regulate immune response.\nSpecifically, CRISPLD2 may interfere with IL1 b-induced cytokine\nproduction and act to reduce immune response via a negative\nfeedback loop that can be activated by IL1 b. This negative\nfeedback loop may also play a role in cytokine level modulation in\nresponse to DEX treatment, as evidenced by the increased levels of\nIL6 observed when both IL1 b and DEX were administered to'),
 Document(id='27e29ea6-1d06-4

### iii. Metadata filtering

Metadata filtering restricts retrieval to documents with selected labels, such as source file or PDF page ranges.


In [7]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
splits = text_splitter.split_documents(study_pdf)
vector_store = InMemoryVectorStore.from_documents(documents=splits, embedding=embedding_model)

first_page_hits = [
    doc for doc in vector_store.similarity_search(QUERY, k=8)
    if doc.metadata.get("page") == 0
]
early_results_hits = [
    doc for doc in vector_store.similarity_search(QUERY, k=8)
    if 1 <= doc.metadata.get("page", -1) <= 4
]
paper_pdf_hits = [
    doc for doc in vector_store.similarity_search(QUERY, k=8)
    if doc.metadata.get("doc_type") == "paper_pdf"
]

df = pd.DataFrame([
    {"filter": "first PDF page", "source": format_sources(first_page_hits)},
    {"filter": "early results pages", "source": format_sources(early_results_hits)},
    {"filter": "paper PDF source", "source": format_sources(paper_pdf_hits)},
])
df.style.set_properties(
    subset=["source"],
    **{"white-space": "pre-wrap", "text-align": "left", "min-width": "650px"},
)


,filter,source
0,first PDF page,None
1,early results pages,../docs/paper.pdf
2,paper PDF source,../docs/paper.pdf


This is a small table comparing the sources found by each filter.

### iv. Hybrid search

Combines semantic search with keyword search. Useful when exact terms matter, like gene names, drug names, IDs, or citations. In fact, this is particularly useful in a scientific context where queries contain precise biomedical terms. 


In [8]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
splits = text_splitter.split_documents(study_pdf)
vector_store = InMemoryVectorStore.from_documents(documents=splits, embedding=embedding_model)

semantic_hits = vector_store.similarity_search(QUERY, k=3)
keyword_hits = [
    doc for doc in splits
    if all(term in doc.page_content for term in KEYWORD_TERMS)
][:3]

hybrid_hits = []
for doc in semantic_hits + keyword_hits:
    if doc not in hybrid_hits:
        hybrid_hits.append(doc)

df = pd.DataFrame([
    {"retrieval": "semantic search", "results": format_contexts(semantic_hits)},
    {"retrieval": "keyword search", "results": format_contexts(keyword_hits)},
    {"retrieval": "hybrid search", "results": format_contexts(hybrid_hits[:3])},
])
df.style.set_properties(
    subset=["results"],
    **{"white-space": "pre-wrap", "text-align": "left", "min-width": "650px"},
)


,retrieval,results
0,semantic search,"1. IL1b responsiveness of two inflammatory genes (i.e. IL6 and IL8), suggesting that CRISPLD2 may regulate immune response. Specifically, CRISPLD2 may interfere with IL1 b-induced cytokine production and act to reduce immune response via a negative feedback loop that can be activated by IL1 b. This negative feedback loop may also play a role in cytokine level modulation in response to DEX treatment, as evidenced by the increased levels of IL6 observed when both IL1 b and DEX were administered to 2. CRISPLD2 knockdown, treatment of ASM cells with IL1 b induced significantly higher expression of IL6 in CRISPLD2-knockdown cells as compared to NT siRNA control cells [Figure 4B], suggesting that CRISPLD2 is an inhibitory modulator of immuno- response in ASM cells. Consistent with this notion, another cytokine’s (i.e. IL8’s) induction by IL1 b was also enhanced by CRISPLD2 knockdown [Figure S6]. To further characterize the effect of CRISPLD2 on immune response, we treated cells with 3. responses by activating other cytokines, we investigated the role of CRISPLD2 in IL1 b-induced expression of other known immune- response genes (i.e. IL6 [32] and IL8 [33]). In ASM cells transfected with CRISPLD2-specific siRNA, CRISPLD2 mRNA expression was decreased by 74% and protein levels decreased by 60% [Figure 4A]. While expression levels of IL6 did not change in response to CRISPLD2 knockdown, treatment of ASM cells with IL1 b induced"
1,keyword search,"1. bronchodilator response among asthma patients in two previously conducted genome-wide association studies. Quantitative RT-PCR and Western blotting showed that dexamethasone treatment significantly increased CRISPLD2 mRNA and protein expression in ASM cells. CRISPLD2 expression was also induced by the inflammatory cytokine IL1 b, and small interfering RNA-mediated knockdown of CRISPLD2 further increased IL1 b-induced expression of IL6 and IL8. Our findings 2. bronchodilator response). Functional experiments showed that in ASM cells, CRISPLD2 mRNA and protein levels changed in response to treatment with a glucocorticoid or proinflammatory cytokine, and that knockdown of CRISPLD2 resulted in increased levels of IL1 b-induced IL6 and IL8 mRNA expression. Results RNA-Seq Transcriptome Profiling of GC-treated Primary Human ASM Cells To identify GC-responsive genes in ASM, we performed RNA- Seq expression profiling of primary ASM cells from four white 3. responses by activating other cytokines, we investigated the role of CRISPLD2 in IL1 b-induced expression of other known immune- response genes (i.e. IL6 [32] and IL8 [33]). In ASM cells transfected with CRISPLD2-specific siRNA, CRISPLD2 mRNA expression was decreased by 74% and protein levels decreased by 60% [Figure 4A]. While expression levels of IL6 did not change in response to CRISPLD2 knockdown, treatment of ASM cells with IL1 b induced"
2,hybrid search,"1. IL1b responsiveness of two inflammatory genes (i.e. IL6 and IL8), suggesting that CRISPLD2 may regulate immune response. Specifically, CRISPLD2 may interfere with IL1 b-induced cytokine production and act to reduce immune response via a negative feedback loop that can be activated by IL1 b. This negative feedback loop may also play a role in cytokine level modulation in response to DEX treatment, as evidenced by the increased levels of IL6 observed when both IL1 b and DEX were administered to 2. CRISPLD2 knockdown, treatment of ASM cells with IL1 b induced significantly higher expression of IL6 in CRISPLD2-knockdown cells as compared to NT siRNA control cells [Figure 4B], suggesting that CRISPLD2 is an inhibitory modulator of immuno- response in ASM cells. Consistent with this notion, another cytokine’s (i.e. IL8’s) induction by IL1 b was also enhanced by CRISPLD2 knockdown [Figure S6]. To further characterize the effect of CRISPLD2 on immune response, we treated cells with 3. responses by activating other cytokines, we investigated the role of CRISPL

Above is a table showing a comparison of three retrieval strategies: semantic, keyword, and hybrid.
- Semantic: passages that the embedding model thinks are conceptually most relevant, even if they do not contain the exact same words as the query
- Keyword: keeps only chunks where every term in KEYWORD_TERMS appears literally in the text
- Hybrid: deduplicated mix of semantic and keyword results

Note: the hybrid row may look identical or very similar to semantic search because semantic results are added first and then the list is cut to 3. If the semantic search already returns 3 unique chunks, the keyword results may not appear in the displayed hybrid output.

## Open ended exercise

Choose a question of your choice on the RNA-seq paper and tweak the RAG parameters we saw in this notebook to reach a desired setup. 

In [ ]:
QUESTION = ""
EMBEDDING_MODEL = ""
CHUNK_SIZE = 0
CHUNK_OVERLAP = 0
RETRIEVAL_K = 0

embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)
splits = text_splitter.split_documents(study_pdf)
vector_store = InMemoryVectorStore.from_documents(documents=splits, embedding=embedding_model)

hits = vector_store.similarity_search(QUESTION, k=RETRIEVAL_K)

pd.DataFrame([
    {"setting": "question", "value": QUESTION},
    {"setting": "embedding model", "value": EMBEDDING_MODEL},
    {"setting": "chunk size", "value": CHUNK_SIZE},
    {"setting": "chunk overlap", "value": CHUNK_OVERLAP},
    {"setting": "retrieved chunks", "value": format_contexts(hits)},
])
